# IMMI - Projet de classification (version exercice guidé)

Ce notebook reprend la structure de ton projet original (segmentation d'IRM cérébrales de nourrissons sur le dataset **iSeg-2017**, via une **cascade de 3 U-Net**) mais avec le code retiré.

**Objectif du projet :** à partir d'images IRM T1 et T2, prédire automatiquement à quelle classe de tissu appartient chaque pixel du cerveau : LCR (liquide céphalo-rachidien / CSF), matière grise (GM) et matière blanche (WM).

**Stratégie (cascade) :**
- **Étage 1** : prédire *cerveau vs fond* (à partir de T1 + T2)
- **Étage 2** : prédire *(GM+WM) vs (CSF+fond)* (à partir de T1 + T2 + prédiction étage 1)
- **Étage 3** : prédire *GM vs le reste* (à partir de T1 + T2 + prédiction étage 1 + prédiction étage 2)

Chaque étage réutilise les prédictions du précédent : ses erreurs se propagent donc en cascade, ce qui explique pourquoi le dice diminue à chaque étage.

Complète les cellules marquées `# TODO`. Les cellules markdown posent les questions auxquelles ton code doit répondre ; garde une trace de tes observations (chiffres, dice, remarques) dans les cellules markdown prévues à cet effet.


## Step 1 - Chargement et affichage des données

**Récupération du dataset (sur Google Drive)**

Monte ton Drive et pointe vers le dossier `iSeg-2017-Training`.

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Bibliothèques utiles
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib  # Pour lire les fichiers médicaux (.hdr/.img)
import os

# TODO : définis le chemin vers ton dataset
data_directory = None  # ex : "/content/drive/MyDrive/IMMI/iSeg-2017-Training"


**Chargement des patients**

Les fichiers sont nommés `subject-{id}-T1.hdr`, `subject-{id}-T2.hdr` et `subject-{id}-label.hdr`.

1. Écris une fonction qui charge le T1, le T2 et la vérité terrain (label) d'un patient donné.
2. Écris une fonction qui affiche la coupe médiane (le long de l'axe z) du T1, du T2 et du label, pour vérifier visuellement que le chargement fonctionne.
3. Affiche les 3 premiers patients.

In [ ]:
def load_patient(patient_id):
    """Charge le T1, le T2 et le label d'un patient à partir de son identifiant.

    Retourne (t1_img, t2_img, label_img) sous forme de tableaux numpy.
    """
    # TODO : construire les chemins de fichiers avec os.path.join et data_directory
    # TODO : charger avec nib.load(...).get_fdata()
    pass


def plot_median_slices(patient_id):
    """Affiche la coupe médiane (axe z) du T1, T2 et label pour un patient."""
    # TODO : appeler load_patient, prendre la coupe au milieu de l'axe z, afficher avec plt.imshow
    pass


# TODO : afficher les 3 premiers patients pour vérifier le chargement


## Step 2 - Normalisation des données

**Pourquoi normaliser ?** Des données non normalisées peuvent empêcher le modèle de converger.

- Normalise le T1 et le T2 avec une **normalisation gaussienne** (soustraire la moyenne, diviser par l'écart-type), calculée uniquement sur les valeurs strictement positives (le fond est à 0 et ne doit pas fausser la moyenne/l'écart-type).
- Ne normalise **pas** les vérités terrain : elles servent de référence (output attendu), pas d'input.
- Divise les 10 patients en trois ensembles : **entraînement**, **validation** et **test**.
  - Entraînement → met à jour les poids du modèle.
  - Validation → surveille l'apprentissage (évite l'overfitting), utilisée pendant l'entraînement mais pas pour mettre à jour les poids.
  - Test → évalue le modèle final, jamais vu pendant l'entraînement.

In [ ]:
def gaussian_normalization(image):
    """Normalise une image en soustrayant la moyenne et en divisant par l'écart-type,
    calculés sur les valeurs strictement positives uniquement.
    """
    # TODO : sélectionner les valeurs > 0
    # TODO : calculer mu et sigma sur ces valeurs
    # TODO : appliquer la normalisation uniquement là où image > 0, garder 0 ailleurs
    pass


# TODO : diviser les identifiants de patients (0 à 9) en train / validation / test
# Combien de patients dans chaque ensemble ? Justifie ton choix dans la cellule markdown suivante.
train_patients = None
val_patients = None
test_patients = None


*Note ici combien de patients tu mets dans chaque ensemble et pourquoi.*

**Vérification** : après normalisation, la moyenne des valeurs positives doit tendre vers 0 et l'écart-type vers 1. Vérifie-le sur un patient.

In [ ]:
# TODO : vérifier la normalisation sur un patient (moyenne ~ 0, écart-type ~ 1)


## Step 3 - Formatage des données en mode 2D

On n'a que 10 patients : c'est très peu pour entraîner un modèle. On va découper chaque volume 3D en coupes 2D le long de l'axe z, ce qui multiplie fortement le nombre d'exemples d'entraînement.

- Pour chaque patient et chaque coupe z, ne garde que les coupes contenant **plus de 100 pixels de tissu cérébral** (label > 0), pour éviter de donner des coupes vides ou quasi vides au modèle.
- Construis X (les entrées : T1 + T2 empilés sur un axe de canaux) et Y (la vérité terrain de la coupe).
- Applique cette fonction aux ensembles train, validation et test.

In [ ]:
def prepare_2d_data(patient_ids):
    """Découpe les volumes 3D des patients donnés en coupes 2D le long de z,
    en ne gardant que les coupes avec suffisamment de tissu cérébral.

    Retourne X (N, H, W, 2) et Y (N, H, W, 1).
    """
    X = []
    Y = []

    for patient_id in patient_ids:
        # TODO : charger et normaliser le patient (attention à l'offset +1 si patient_id part de 0)

        # TODO : parcourir les coupes en z, filtrer selon le nombre de pixels de tissu (> 100),
        # empiler T1 et T2 dans X, ajouter le label correspondant dans Y
        pass

    # TODO : convertir X et Y en tableaux numpy avant de les retourner
    return np.array(X), np.array(Y)


# TODO : appliquer prepare_2d_data aux trois ensembles de patients
X_train, Y_train = None, None
X_val, Y_val = None, None
X_test, Y_test = None, None


Vérifie à nouveau la normalisation, mais cette fois sur les données 2D préparées (histogramme + moyenne/écart-type de T1 et T2).

In [ ]:
# TODO : tracer l'histogramme de X_train[:,:,:,0] (T1) et X_train[:,:,:,1] (T2)
# TODO : afficher moyenne et écart-type de T1 et T2


*Que remarques-tu ? La normalisation te semble-t-elle correcte (moyenne ~0, écart-type ~1) ? Est-ce que ça a un impact sur la suite ?*

## Step 4 - Un premier entraînement de réseau !

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


**Le U-Net**

Le U-Net est une architecture encodeur-décodeur avec des connexions ("skip connections") entre les couches symétriques de l'encodeur et du décodeur, très utilisée pour la segmentation d'images.

- Construis un encodeur : blocs `Conv2D → Conv2D → MaxPooling2D`, avec un nombre de filtres qui double à chaque bloc (par ex. 64, 128, 256, ...).
- Construis un décodeur symétrique : blocs `UpSampling2D → concatenate (avec la couche encodeur correspondante) → Conv2D → Conv2D`.
- Termine par une `Conv2D` de sortie à 1 filtre avec une activation `sigmoid` (segmentation binaire à cet étage : cerveau vs fond).

In [ ]:
def unet(input_size=(144, 192, 2)):
    """U-Net simple pour segmentation binaire (2 canaux d'entrée : T1, T2)."""
    inputs = Input(input_size)

    # TODO : encodeur (Conv2D + Conv2D + MaxPooling2D, filtres qui doublent)

    # TODO : bottleneck

    # TODO : décodeur symétrique (UpSampling2D + concatenate + Conv2D + Conv2D)

    # TODO : couche de sortie Conv2D(1, 1, activation='sigmoid')

    model = None  # Model(inputs=inputs, outputs=...)
    return model


**Entraînement**

- Compile le modèle (optimiseur `Adam`, loss adaptée à une segmentation binaire, par ex. `binary_crossentropy`).
- Ajoute un `EarlyStopping` (surveille le loss de validation, patience de 5) pour arrêter l'entraînement au bon moment.
- Ajoute un `ReduceLROnPlateau` pour réduire le learning rate en cas de plateau du loss de validation.
- Entraîne le modèle sur `X_train`/`Y_train`, avec `X_val`/`Y_val` en validation.

In [ ]:
# TODO : instancier le modèle, le compiler, définir les callbacks et l'entraîner
model = unet()
# model.compile(...)
# callbacks = [EarlyStopping(...), ReduceLROnPlateau(...)]
# history = model.fit(...)


On sauvegarde les poids du modèle sur Drive afin de pouvoir le réutiliser plus tard sans le ré-entraîner.

In [ ]:
saved_weights_path_1 = '/content/drive/MyDrive/IMMI/unet_weights.weights.h5'
model.save_weights(saved_weights_path_1)


## Step 5 - Finir la première étape de la cascade

**Évaluation**

Le dice mesure la similarité géométrique entre la prédiction et la vérité terrain : `2 * intersection / (somme des deux ensembles)`.

- Écris la fonction de dice.
- Écris une fonction qui évalue le modèle sur les patients de test : pour chaque patient, prédis chaque coupe z, reconstruis le volume, binarise, puis calcule le dice par rapport au label.
- Calcule le dice moyen sur les patients de test.

In [ ]:
def dice_coefficient(y_true, y_pred, epsilon=1e-6):
    # TODO : implémenter la formule du dice
    pass


def evaluate_on_test(model, test_patients):
    """Évalue le modèle sur les patients de test et retourne la liste des dice scores."""
    dice_scores = []

    for patient_id in test_patients:
        # TODO : charger + normaliser le patient
        # TODO : prédire chaque coupe z, reconstruire le volume prédit
        # TODO : binariser la prédiction et calculer le dice par rapport au label
        pass

    return dice_scores


# TODO : appeler evaluate_on_test et afficher le dice moyen


*Quel dice moyen obtiens-tu ? Est-ce suffisant pour passer à la suite ?*

**Vérification visuelle**

Choisis un patient de test et une coupe aléatoire contenant suffisamment de tissu cérébral. Affiche T1, T2, la vérité terrain, la prédiction binarisée, et la prédiction brute (non binarisée, pour voir la confiance du modèle).

In [ ]:
def visualize_random_test_prediction(model, test_patients):
    # TODO : choisir un patient et une coupe valide, prédire, puis afficher avec plt.imshow
    # (T1, T2, vérité terrain, prédiction binarisée, prédiction brute)
    pass


# TODO : appeler la fonction sur un patient de test


## Step 6 - Programmation de l'étage 2 de la cascade

L'étage 2 prédit maintenant *matière blanche + matière grise* (sans le CSF). Il faut donc :

- Construire une nouvelle vérité terrain binaire : 1 là où le label correspond à GM ou WM, 0 pour le CSF et le fond.
- Utiliser en entrée le T1, le T2, **et** la prédiction de l'étage 1 (empilés sur l'axe des canaux → 3 canaux au lieu de 2).
- Réutiliser la même architecture U-Net, adaptée à 3 canaux d'entrée.

In [ ]:
def prepare_second_stage_data(patient_ids, first_stage_model):
    """Prépare X (T1, T2, pred étage 1) et Y (GM+WM binaire) pour l'étage 2."""
    X = []
    Y = []

    for patient_id in patient_ids:
        # TODO : charger + normaliser le patient
        # TODO : obtenir la prédiction de first_stage_model pour tout le volume
        # TODO : construire la vérité terrain binaire GM+WM à partir du label
        # TODO : empiler T1, T2, pred étage 1 par coupe (en filtrant comme à l'étape 3)
        pass

    return np.array(X), np.array(Y)


def unet_second_stage(input_size=(144, 192, 3)):
    """Même architecture que unet(), adaptée à 3 canaux d'entrée."""
    # TODO : soit dupliquer unet() en changeant input_size, soit rendre unet() générique
    pass


# TODO : préparer les données de l'étage 2 (train/val/test), instancier, compiler et entraîner model_2


In [ ]:
saved_weights_path_2 = '/content/drive/MyDrive/IMMI/unet_weights2.weights.h5'
# TODO : model_2.save_weights(saved_weights_path_2)


Évalue l'étage 2 comme l'étage 1 (dice sur les patients de test) et affiche un exemple visuel.

In [ ]:
def evaluate_second_stage(model_1, model_2, test_patients):
    # TODO : comme evaluate_on_test, mais en chaînant model_1 -> model_2
    pass


def visualize_second_stage_results(model_1, model_2, test_patients):
    # TODO : afficher un exemple (T1, T2, pred étage 1, VT étage 2, prédiction étage 2)
    pass


*Quel dice obtiens-tu à cet étage ? Est-il meilleur ou moins bon qu'à l'étage 1 ? Pourquoi, selon toi ?*

## Step 7 - Programmation de l'étage 3 de la cascade et regroupement en données finales

L'étage 3 prédit la **matière grise (GM)** seule. Les entrées sont maintenant T1, T2, prédiction étage 1 et prédiction étage 2 (4 canaux). La vérité terrain doit à nouveau être adaptée (binaire : GM ou non).

In [ ]:
def prepare_third_stage_data(patient_ids, model_1, model_2):
    """Prépare X (T1, T2, pred étage 1, pred étage 2) et Y (GM binaire) pour l'étage 3."""
    X = []
    Y = []

    for patient_id in patient_ids:
        # TODO : charger + normaliser le patient
        # TODO : obtenir les prédictions des étages 1 et 2 pour tout le volume
        # TODO : construire la vérité terrain binaire GM
        # TODO : empiler T1, T2, pred1, pred2 par coupe (avec le même filtrage qu'avant)
        pass

    return np.array(X), np.array(Y)


def unet_third_stage(input_size=(144, 192, 4)):
    """Même architecture, adaptée à 4 canaux d'entrée."""
    # TODO
    pass


# TODO : préparer les données, instancier, compiler et entraîner model_3


In [ ]:
saved_weights_path_3 = '/content/drive/MyDrive/IMMI/unet_weights3.weights.h5'
# TODO : model_3.save_weights(saved_weights_path_3)


**Évaluation complète de la cascade**

Écris une fonction qui, pour chaque patient de test, fait passer les données à travers les 3 étages, et calcule le dice de chaque étage. Attention aux dimensions (`.squeeze()` peut être utile si tu as des erreurs de shape).

In [ ]:
def evaluate_full_cascade(model_1, model_2, model_3, test_patients):
    """Retourne, pour chaque patient de test, les prédictions des 3 étages et leurs dice scores."""
    results = []

    for patient_id in test_patients:
        # TODO : charger + normaliser le patient
        # TODO : prédire étage 1, étage 2, étage 3, coupe par coupe
        # TODO : calculer un dice par étage et stocker le tout (predictions + dice) dans `results`
        pass

    return results


# TODO : appeler evaluate_full_cascade et afficher le dice moyen de chaque étage


*Note ici tes dice pour les 3 étages. Sont-ils décroissants d'un étage à l'autre ? À ton avis, pourquoi les erreurs s'accumulent-elles en cascade ?*

**Visualisation et regroupement final**

- Affiche, pour un patient donné, T1, T2, la vérité terrain et les prédictions des 3 étages côte à côte.
- Combine les prédictions des 3 étages en une seule carte multi-classes (fond / CSF / GM / WM), en utilisant les valeurs de label du dataset iSeg (typiquement fond=0, CSF=10, GM=150, WM=250).
- Si tes couleurs ne correspondent pas à ce qui est attendu après le regroupement, écris une fonction de correction qui fait correspondre les valeurs obtenues aux valeurs attendues.

In [ ]:
def show_patient_slice(patient_id, cascade_results):
    # TODO : afficher une coupe de T1/T2/label et les prédictions des 3 étages pour un patient
    pass


def combine_cascade_predictions(cascade_results):
    """Combine pred1/pred2/pred3 en une carte multi-classes (fond/CSF/GM/WM)."""
    # TODO : réfléchis à la logique de combinaison (ex : partir du fond, ajouter CSF là où pred1=1
    # et pred2=0, ajouter WM là où pred2=1 et pred3=0, ajouter GM là où pred3=1)
    pass


def show_patient_results(patient_id, cascade_results):
    # TODO : affichage final comparant la prédiction combinée à la vérité terrain
    pass


*À quel étage observes-tu la plus grande différence entre prédiction et vérité terrain ? Est-ce cohérent avec les dice calculés plus haut ?*

## Pour aller plus loin

Une fois la cascade complète et évaluée, réfléchis à des pistes d'amélioration : taille du dataset (10 patients seulement !), choix des hyperparamètres (batch size, learning rate, architecture), data augmentation, autres architectures de segmentation, etc.

*Note ici tes propres pistes d'amélioration, à la lumière de tes résultats.*